# Data Fusion 2026 — dual boosting notebook

Этот ноутбук оставляет сильный causal feature engineering и честную temporal-валидацию из улучшенного baseline,
но упрощает модельную часть до компактного ансамбля из двух моделей:

- один `CatBoost`;
- один `LightGBM`;
- rank-blend по temporal holdout.

Такой пайплайн проще запускать и тюнить, чем 4-head ансамбль, при этом он сохраняет хорошую устойчивость к утечкам и temporal drift.


In [1]:

from pathlib import Path
import gc
import json
import os
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb

from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier, Pool
from scipy.optimize import minimize
from scipy.special import softmax
from scipy.stats import rankdata

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(200)

DATA_DIR  = Path("data")
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

NEG_SAMPLE_MOD_RECENT = 10
NEG_SAMPLE_MOD_OLD    = 30
NEG_SAMPLE_BORDER_STR = "2025-04-01 00:00:00"

VAL_START            = pd.Timestamp("2025-05-01")
RECENCY_WEIGHT_START = pd.Timestamp("2025-02-01")

RANDOM_SEED            = 67
FEATURE_CACHE_TAG      = "v11_robust_features"
GLOBAL_CACHE_TAG       = "v3_honest_eval"
FORCE_REBUILD_FEATURES = False
FORCE_REBUILD_GLOBALS  = False
USE_GPU                = True
RETRAIN_ON_FULL        = True
REFIT_ITER_MULT        = 1.10

np.random.seed(RANDOM_SEED)

print("LightGBM version:", lgb.__version__)
print("DATA_DIR:", DATA_DIR.resolve())
print("CACHE_DIR:", CACHE_DIR.resolve())


LightGBM version: 4.6.0
DATA_DIR: /workspace/task1/data
CACHE_DIR: /workspace/task1/cache


## 1. Схема данных и список фич

In [2]:

BASE_COLS = [
    "customer_id", "event_id", "event_dttm", "event_type_nm", "event_desc",
    "channel_indicator_type", "channel_indicator_sub_type", "operaton_amt", "currency_iso_cd",
    "mcc_code", "pos_cd",
    "accept_language", "browser_language",
    "timezone", "session_id", "operating_system_type",
    "battery", "device_system_version", "screen_size", "developer_tools",
    "phone_voip_call_state", "web_rdp_connection", "compromised",
]

FINAL_FEATURE_COLS = [
    # категориальные идентификаторы
    "customer_id", "event_type_nm", "event_desc", "channel_indicator_type",
    "channel_indicator_sub_type", "currency_iso_cd", "mcc_code_i", "pos_cd", "timezone",
    "operating_system_type", "phone_voip_call_state", "web_rdp_connection",
    "developer_tools_i", "compromised_i",
    "accept_language_i", "browser_language_i",
    "device_fp_i",

    # amount + missingness
    "amt", "amt_abs", "amt_log_abs", "amt_is_negative", "amt_bucket",
    "amt_missing", "currency_missing", "mcc_missing", "pos_missing",
    "accept_language_missing", "browser_language_missing", "timezone_missing",
    "session_id_missing", "operating_system_missing", "battery_missing",
    "device_system_version_missing", "screen_size_missing", "developer_tools_missing",
    "phone_voip_missing", "web_rdp_missing", "compromised_missing",

    # временные
    "hour", "weekday", "day", "is_weekend", "is_night",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "month_sin", "month_cos",

    # устройство
    "battery_pct", "os_ver_major", "screen_w", "screen_h", "screen_pixels", "screen_ratio",

    # возраст аккаунта и активность
    "days_since_first_event", "cust_active_days", "avg_prev_events_per_active_day",

    # клиентская история объёмов
    "cust_prev_events", "cust_prev_amt_mean", "cust_prev_amt_std", "cust_prev_max_amt",
    "amt_vs_personal_max",
    "sec_since_prev_event", "days_since_prev_event",
    "amt_delta_prev", "amt_zscore",

    # сессионные
    "sec_since_session_start", "session_amt_before", "cnt_prev_same_session",

    # velocity-счётчики
    "cnt_prev_same_type", "cnt_prev_same_desc", "cnt_prev_same_mcc",
    "cnt_prev_same_subtype", "cnt_prev_same_channel_type",
    "cnt_prev_same_currency", "cnt_prev_same_device",
    "sec_since_prev_same_type", "sec_since_prev_same_desc",
    "sec_since_prev_same_channel_type", "sec_since_prev_same_currency", "sec_since_prev_same_device",
    "events_before_today", "cnt_events_this_hour",

    # флаги «впервые»
    "is_new_event_type", "is_new_event_desc", "is_new_channel_sub",
    "is_new_channel_type", "is_new_mcc", "is_new_currency",
    "is_new_device_fp", "is_first_in_session",

    # rolling
    "amt_sum_last_1h", "cnt_last_1h",
    "amt_sum_last_24h", "cnt_last_24h", "max_amt_last_24h",
    "amt_vs_1h_sum", "amt_vs_24h_sum",

    # история меток клиента
    "cust_prev_red_lbl_cnt", "cust_prev_yellow_lbl_cnt", "cust_prev_labeled_cnt",
    "cust_prev_red_lbl_rate", "cust_prev_yellow_lbl_rate", "cust_prev_susp_lbl_rate",
    "cust_prev_any_red_flag", "cust_prev_any_yellow_flag",
    "sec_since_prev_red_lbl", "sec_since_prev_yellow_lbl",

    # fine-grained label history
    "cnt_prev_labeled_same_desc", "cnt_prev_red_same_desc_lbl", "cnt_prev_yellow_same_desc_lbl",
    "red_rate_prev_same_desc_lbl",
    "cnt_prev_red_same_channel", "cnt_prev_labeled_same_channel", "red_rate_prev_same_channel",
    "cnt_prev_red_same_type_lbl", "cnt_prev_labeled_same_type_lbl", "red_rate_prev_same_type",
]

CAT_COLS = [
    "customer_id", "event_type_nm", "event_desc", "channel_indicator_type",
    "channel_indicator_sub_type", "currency_iso_cd", "mcc_code_i", "pos_cd",
    "timezone", "operating_system_type", "phone_voip_call_state", "web_rdp_connection",
    "developer_tools_i", "compromised_i",
    "accept_language_i", "browser_language_i",
    "device_fp_i",
]

META_COLS = [
    "event_id", "period", "event_ts",
    "is_train_sample", "is_test",
    "train_target_raw", "target_bin",
]

MODEL_DROP_COLS = {
    "target", "keep_green", "event_date", "event_hour_trunc",
    "month", "event_day_number",
    "session_id",
}

RISKY_LABEL_FEATURES = {
    "cnt_prev_labeled_same_desc", "cnt_prev_red_same_desc_lbl", "cnt_prev_yellow_same_desc_lbl",
    "red_rate_prev_same_desc_lbl",
    "cnt_prev_red_same_channel", "cnt_prev_labeled_same_channel", "red_rate_prev_same_channel",
    "cnt_prev_red_same_type_lbl", "cnt_prev_labeled_same_type_lbl", "red_rate_prev_same_type",
}

RISKY_PRIOR_SUFFIXES = (
    "_red_rate_all",
    "_labeled_rate_all",
    "_red_share_labeled",
)

labels_lf = pl.scan_parquet(DATA_DIR / "train_labels.parquet")
labels_df = pl.read_parquet(DATA_DIR / "train_labels.parquet")
print("Labels:", labels_df.shape)


Labels: (87514, 3)


## 2. Базовый feature builder

In [3]:

def _period_frames_for_part(part_id: int) -> pl.LazyFrame:
    custs_lf = (
        pl.scan_parquet(DATA_DIR / f"pretrain_part_{part_id}.parquet")
        .select("customer_id").unique()
    )
    def _load(path, period):
        cols = [c for c in BASE_COLS if c in pl.scan_parquet(path).columns]
        return pl.scan_parquet(path).select(cols).with_columns(pl.lit(period).alias("period"))

    pretrain_lf = _load(DATA_DIR / f"pretrain_part_{part_id}.parquet", "pretrain")
    train_lf    = _load(DATA_DIR / f"train_part_{part_id}.parquet", "train")
    pretest_lf  = (
        _load(DATA_DIR / "pretest.parquet", "pretest")
        .join(custs_lf, on="customer_id", how="inner")
    )
    test_lf = (
        _load(DATA_DIR / "test.parquet", "test")
        .join(custs_lf, on="customer_id", how="inner")
    )
    return pl.concat([pretrain_lf, train_lf, pretest_lf, test_lf], how="diagonal_relaxed")


def build_features_for_part(part_id: int, force: bool = False) -> Path:
    out_path = CACHE_DIR / f"features_{FEATURE_CACHE_TAG}_part_{part_id}.parquet"
    if out_path.exists() and (not force):
        print(f"[part {part_id}] use cache -> {out_path.name}")
        return out_path

    print(f"[part {part_id}] building features...")
    lf = _period_frames_for_part(part_id)

    lf = (
        lf.with_columns([
            pl.col("event_dttm").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False).alias("event_ts"),
            pl.col("operaton_amt").cast(pl.Float64).alias("amt"),
            pl.col("session_id").cast(pl.Int64, strict=False).fill_null(-1).alias("session_id"),
            pl.col("event_type_nm").cast(pl.Int32, strict=False).fill_null(-1).alias("event_type_nm"),
            pl.col("event_desc").cast(pl.Int32, strict=False).fill_null(-1).alias("event_desc"),
            pl.col("channel_indicator_type").cast(pl.Int16, strict=False).fill_null(-1).alias("channel_indicator_type"),
            pl.col("channel_indicator_sub_type").cast(pl.Int16, strict=False).fill_null(-1).alias("channel_indicator_sub_type"),
            pl.col("currency_iso_cd").cast(pl.Int16, strict=False).fill_null(-1).alias("currency_iso_cd"),
            pl.col("pos_cd").cast(pl.Int16, strict=False).fill_null(-1).alias("pos_cd"),
            pl.col("timezone").cast(pl.Int32, strict=False).fill_null(-1).alias("timezone"),
            pl.col("operating_system_type").cast(pl.Int16, strict=False).fill_null(-1).alias("operating_system_type"),
            pl.col("phone_voip_call_state").cast(pl.Int8, strict=False).fill_null(-1).alias("phone_voip_call_state"),
            pl.col("web_rdp_connection").cast(pl.Int8, strict=False).fill_null(-1).alias("web_rdp_connection"),
            pl.col("mcc_code").cast(pl.Int32, strict=False).fill_null(-1).alias("mcc_code_i"),
            pl.col("battery").str.extract(r"(\d{1,3})", 1).cast(pl.Int16, strict=False).fill_null(-1).alias("battery_pct"),
            pl.col("device_system_version").str.extract(r"^(\d+)", 1).cast(pl.Int16, strict=False).fill_null(-1).alias("os_ver_major"),
            pl.col("screen_size").str.extract(r"^(\d+)", 1).cast(pl.Int16, strict=False).fill_null(-1).alias("screen_w"),
            pl.col("screen_size").str.extract(r"x(\d+)$", 1).cast(pl.Int16, strict=False).fill_null(-1).alias("screen_h"),
            pl.col("developer_tools").cast(pl.Int8, strict=False).fill_null(-1).alias("developer_tools_i"),
            pl.col("compromised").cast(pl.Int8, strict=False).fill_null(-1).alias("compromised_i"),
            pl.col("accept_language").cast(pl.Int32, strict=False).fill_null(-1).alias("accept_language_i"),
            pl.col("browser_language").cast(pl.Int32, strict=False).fill_null(-1).alias("browser_language_i"),

            pl.col("operaton_amt").is_null().cast(pl.Int8).alias("amt_missing"),
            pl.col("currency_iso_cd").is_null().cast(pl.Int8).alias("currency_missing"),
            pl.col("mcc_code").is_null().cast(pl.Int8).alias("mcc_missing"),
            pl.col("pos_cd").is_null().cast(pl.Int8).alias("pos_missing"),
            pl.col("accept_language").is_null().cast(pl.Int8).alias("accept_language_missing"),
            pl.col("browser_language").is_null().cast(pl.Int8).alias("browser_language_missing"),
            pl.col("timezone").is_null().cast(pl.Int8).alias("timezone_missing"),
            pl.col("session_id").is_null().cast(pl.Int8).alias("session_id_missing"),
            pl.col("operating_system_type").is_null().cast(pl.Int8).alias("operating_system_missing"),
            pl.col("battery").is_null().cast(pl.Int8).alias("battery_missing"),
            pl.col("device_system_version").is_null().cast(pl.Int8).alias("device_system_version_missing"),
            pl.col("screen_size").is_null().cast(pl.Int8).alias("screen_size_missing"),
            pl.col("developer_tools").is_null().cast(pl.Int8).alias("developer_tools_missing"),
            pl.col("phone_voip_call_state").is_null().cast(pl.Int8).alias("phone_voip_missing"),
            pl.col("web_rdp_connection").is_null().cast(pl.Int8).alias("web_rdp_missing"),
            pl.col("compromised").is_null().cast(pl.Int8).alias("compromised_missing"),
        ])
        .drop([
            "event_dttm", "operaton_amt", "mcc_code", "battery", "device_system_version",
            "screen_size", "developer_tools", "compromised", "accept_language", "browser_language",
        ])
        .sort(["customer_id", "event_ts", "event_id"])
    )

    lf = lf.with_columns([
        (
            pl.col("screen_w").cast(pl.Int64) * 100_000_000
            + pl.col("screen_h").cast(pl.Int64) * 100_000
            + pl.col("operating_system_type").cast(pl.Int64) * 1000
            + pl.col("accept_language_i").cast(pl.Int64) % 1000
        ).alias("device_fp_i"),
    ])

    lf = lf.join(labels_lf, on="event_id", how="left")
    lf = lf.with_columns([
        pl.when(pl.col("period") == "train")
          .then(pl.when(pl.col("target").is_null()).then(pl.lit(-1)).otherwise(pl.col("target")))
          .otherwise(pl.lit(None))
          .alias("train_target_raw")
    ])

    border_expr = pl.lit(NEG_SAMPLE_BORDER_STR).str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
    lf = lf.with_columns([
        ((pl.col("period") == "train") &
         (pl.col("train_target_raw") == -1) &
         (((pl.col("event_ts") >= border_expr) & ((pl.struct(["event_id", "customer_id"]).hash(seed=RANDOM_SEED) % NEG_SAMPLE_MOD_RECENT) == 0)) |
          ((pl.col("event_ts") < border_expr)  & ((pl.struct(["event_id", "customer_id"]).hash(seed=RANDOM_SEED + 17) % NEG_SAMPLE_MOD_OLD) == 0))))
          .alias("keep_green")
    ])
    lf = lf.with_columns([
        ((pl.col("period") == "train") & ((pl.col("train_target_raw") != -1) | pl.col("keep_green"))).alias("is_train_sample"),
        (pl.col("period") == "test").alias("is_test"),
        pl.col("event_ts").dt.hour().cast(pl.Int8).alias("hour"),
        pl.col("event_ts").dt.weekday().cast(pl.Int8).alias("weekday"),
        pl.col("event_ts").dt.day().cast(pl.Int8).alias("day"),
        pl.col("event_ts").dt.month().cast(pl.Int8).alias("month"),
        (pl.col("event_ts").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
        (pl.col("event_ts").dt.hour().is_in([0, 1, 2, 3, 4, 5])).cast(pl.Int8).alias("is_night"),
        (pl.col("event_ts").dt.epoch("s") // 86400).cast(pl.Int32).alias("event_day_number"),
        pl.col("event_ts").dt.date().alias("event_date"),
        pl.col("event_ts").dt.truncate("1h").alias("event_hour_trunc"),
        pl.col("amt").abs().cast(pl.Float32).alias("amt_abs"),
        pl.col("amt").abs().log1p().cast(pl.Float32).alias("amt_log_abs"),
        (pl.col("amt").abs().log1p() * 4.0).floor().clip(0, 63).cast(pl.Int16).alias("amt_bucket"),
        (pl.col("amt") < 0).cast(pl.Int8).alias("amt_is_negative"),
        (pl.col("screen_w").cast(pl.Int32) * pl.col("screen_h").cast(pl.Int32)).alias("screen_pixels"),
        pl.when((pl.col("screen_h") > 0) & (pl.col("screen_w") > 0))
          .then(pl.col("screen_w").cast(pl.Float32) / pl.col("screen_h").cast(pl.Float32))
          .otherwise(0.0).alias("screen_ratio"),
        (((pl.col("event_ts").dt.hour().cast(pl.Float32) / 24.0) * (2.0 * np.pi)).sin()).cast(pl.Float32).alias("hour_sin"),
        (((pl.col("event_ts").dt.hour().cast(pl.Float32) / 24.0) * (2.0 * np.pi)).cos()).cast(pl.Float32).alias("hour_cos"),
        (((pl.col("event_ts").dt.weekday().cast(pl.Float32) / 7.0) * (2.0 * np.pi)).sin()).cast(pl.Float32).alias("weekday_sin"),
        (((pl.col("event_ts").dt.weekday().cast(pl.Float32) / 7.0) * (2.0 * np.pi)).cos()).cast(pl.Float32).alias("weekday_cos"),
        (((pl.col("event_ts").dt.month().cast(pl.Float32) / 12.0) * (2.0 * np.pi)).sin()).cast(pl.Float32).alias("month_sin"),
        (((pl.col("event_ts").dt.month().cast(pl.Float32) / 12.0) * (2.0 * np.pi)).cos()).cast(pl.Float32).alias("month_cos"),
    ])

    lf = lf.with_columns([
        pl.col("event_ts").min().over("customer_id").alias("_cust_first_ts"),
    ])
    lf = lf.with_columns([
        ((pl.col("event_ts") - pl.col("_cust_first_ts")).dt.total_seconds() // 86400)
          .cast(pl.Int32).alias("days_since_first_event"),
    ])
    lf = lf.drop("_cust_first_ts")

    lf = lf.with_columns([
        pl.col("event_ts").min().over(["customer_id", "session_id"]).alias("_session_first_ts"),
        pl.col("amt").cum_sum().over(["customer_id", "session_id"]).alias("_session_cum_amt"),
    ])
    lf = lf.with_columns([
        ((pl.col("event_ts") - pl.col("_session_first_ts")).dt.total_seconds())
          .cast(pl.Int32).alias("sec_since_session_start"),
        (pl.col("_session_cum_amt") - pl.col("amt")).cast(pl.Float32).alias("session_amt_before"),
    ])
    lf = lf.drop(["_session_first_ts", "_session_cum_amt"])

    lf = lf.with_columns([
        ((pl.col("period") == "train") & (pl.col("train_target_raw") == 1)).cast(pl.Int8).alias("is_red_lbl"),
        ((pl.col("period") == "train") & (pl.col("train_target_raw") == 0)).cast(pl.Int8).alias("is_yellow_lbl"),
    ])
    lf = lf.with_columns([
        (pl.col("is_red_lbl") + pl.col("is_yellow_lbl")).cast(pl.Int8).alias("is_labeled_fb")
    ])

    lf = lf.with_columns([
        pl.cum_count("event_id").over("customer_id").cast(pl.Int32).alias("cust_event_idx"),
        pl.col("amt").cum_sum().over("customer_id").alias("cust_cum_amt"),
        (pl.col("amt") * pl.col("amt")).cum_sum().over("customer_id").alias("cust_cum_amt_sq"),
        pl.col("amt").cum_max().over("customer_id").alias("cust_cum_max_amt"),
        pl.col("event_ts").shift(1).over("customer_id").alias("prev_event_ts"),
        pl.col("event_date").shift(1).over("customer_id").alias("prev_event_date"),
        pl.col("amt").shift(1).over("customer_id").alias("prev_amt"),
        (pl.cum_count("event_id").over(["customer_id", "event_type_nm"]) - 1).cast(pl.Int16).alias("cnt_prev_same_type"),
        (pl.cum_count("event_id").over(["customer_id", "event_desc"]) - 1).cast(pl.Int16).alias("cnt_prev_same_desc"),
        (pl.cum_count("event_id").over(["customer_id", "mcc_code_i"]) - 1).cast(pl.Int16).alias("cnt_prev_same_mcc"),
        (pl.cum_count("event_id").over(["customer_id", "channel_indicator_sub_type"]) - 1).cast(pl.Int16).alias("cnt_prev_same_subtype"),
        (pl.cum_count("event_id").over(["customer_id", "channel_indicator_type"]) - 1).cast(pl.Int16).alias("cnt_prev_same_channel_type"),
        (pl.cum_count("event_id").over(["customer_id", "currency_iso_cd"]) - 1).cast(pl.Int16).alias("cnt_prev_same_currency"),
        (pl.cum_count("event_id").over(["customer_id", "device_fp_i"]) - 1).cast(pl.Int16).alias("cnt_prev_same_device"),
        (pl.cum_count("event_id").over(["customer_id", "session_id"]) - 1).cast(pl.Int16).alias("cnt_prev_same_session"),
        pl.col("event_ts").shift(1).over(["customer_id", "event_type_nm"]).alias("prev_same_type_ts"),
        pl.col("event_ts").shift(1).over(["customer_id", "event_desc"]).alias("prev_same_desc_ts"),
        pl.col("event_ts").shift(1).over(["customer_id", "channel_indicator_type"]).alias("prev_same_channel_type_ts"),
        pl.col("event_ts").shift(1).over(["customer_id", "currency_iso_cd"]).alias("prev_same_currency_ts"),
        pl.col("event_ts").shift(1).over(["customer_id", "device_fp_i"]).alias("prev_same_device_ts"),
        pl.col("is_red_lbl").cum_sum().over("customer_id").cast(pl.Int32).alias("cust_red_lbl_cum"),
        pl.col("is_yellow_lbl").cum_sum().over("customer_id").cast(pl.Int32).alias("cust_yellow_lbl_cum"),
        pl.col("is_labeled_fb").cum_sum().over("customer_id").cast(pl.Int32).alias("cust_labeled_fb_cum"),
        pl.col("is_red_lbl").cum_sum().over(["customer_id", "event_desc"]).cast(pl.Int16).alias("desc_red_lbl_cum"),
        pl.col("is_yellow_lbl").cum_sum().over(["customer_id", "event_desc"]).cast(pl.Int16).alias("desc_yellow_lbl_cum"),
        pl.col("is_labeled_fb").cum_sum().over(["customer_id", "event_desc"]).cast(pl.Int16).alias("desc_labeled_fb_cum"),
        pl.col("is_red_lbl").cum_sum().over(["customer_id", "channel_indicator_type"]).cast(pl.Int16).alias("chan_red_lbl_cum"),
        pl.col("is_labeled_fb").cum_sum().over(["customer_id", "channel_indicator_type"]).cast(pl.Int16).alias("chan_labeled_fb_cum"),
        pl.col("is_red_lbl").cum_sum().over(["customer_id", "event_type_nm"]).cast(pl.Int16).alias("type_red_lbl_cum"),
        pl.col("is_labeled_fb").cum_sum().over(["customer_id", "event_type_nm"]).cast(pl.Int16).alias("type_labeled_fb_cum"),
        pl.when(pl.col("is_red_lbl") == 1).then(pl.col("event_ts")).otherwise(None).alias("red_lbl_ts"),
        pl.when(pl.col("is_yellow_lbl") == 1).then(pl.col("event_ts")).otherwise(None).alias("yellow_lbl_ts"),
    ])

    lf = lf.with_columns([
        (pl.cum_count("event_id").over(["customer_id", "event_hour_trunc"]) - 1).cast(pl.Int16).alias("cnt_events_this_hour"),
    ])

    lf = lf.with_columns([
        pl.col("red_lbl_ts").shift(1).over("customer_id").alias("_red_shifted"),
        pl.col("yellow_lbl_ts").shift(1).over("customer_id").alias("_yellow_shifted"),
    ])
    lf = lf.with_columns([
        pl.col("_red_shifted").forward_fill().over("customer_id").alias("prev_red_lbl_ts"),
        pl.col("_yellow_shifted").forward_fill().over("customer_id").alias("prev_yellow_lbl_ts"),
    ])
    lf = lf.drop(["_red_shifted", "_yellow_shifted"])

    lf = lf.with_columns([
        (pl.col("cust_event_idx") - 1).cast(pl.Int32).alias("cust_prev_events"),
        pl.when(pl.col("cust_event_idx") > 1)
          .then((pl.col("cust_cum_amt") - pl.col("amt")) / (pl.col("cust_event_idx") - 1))
          .otherwise(0.0).cast(pl.Float32).alias("cust_prev_amt_mean"),
        pl.col("cust_cum_max_amt").shift(1).over("customer_id").fill_null(0.0).cast(pl.Float32).alias("cust_prev_max_amt"),
        pl.when(pl.col("prev_event_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_event_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_event"),
        (pl.col("amt") - pl.col("prev_amt").fill_null(0.0)).cast(pl.Float32).alias("amt_delta_prev"),
        pl.when(pl.col("prev_same_type_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_same_type_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_same_type"),
        pl.when(pl.col("prev_same_desc_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_same_desc_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_same_desc"),
        pl.when(pl.col("prev_same_channel_type_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_same_channel_type_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_same_channel_type"),
        pl.when(pl.col("prev_same_currency_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_same_currency_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_same_currency"),
        pl.when(pl.col("prev_same_device_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_same_device_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_same_device"),
        (pl.cum_count("event_id").over(["customer_id", "event_date"]) - 1).cast(pl.Int16).alias("events_before_today"),
        (pl.col("cust_red_lbl_cum") - pl.col("is_red_lbl")).cast(pl.Int32).alias("cust_prev_red_lbl_cnt"),
        (pl.col("cust_yellow_lbl_cum") - pl.col("is_yellow_lbl")).cast(pl.Int32).alias("cust_prev_yellow_lbl_cnt"),
        (pl.col("cust_labeled_fb_cum") - pl.col("is_labeled_fb")).cast(pl.Int32).alias("cust_prev_labeled_cnt"),
        (pl.col("desc_labeled_fb_cum") - pl.col("is_labeled_fb")).cast(pl.Int16).alias("cnt_prev_labeled_same_desc"),
        (pl.col("desc_red_lbl_cum") - pl.col("is_red_lbl")).cast(pl.Int16).alias("cnt_prev_red_same_desc_lbl"),
        (pl.col("desc_yellow_lbl_cum") - pl.col("is_yellow_lbl")).cast(pl.Int16).alias("cnt_prev_yellow_same_desc_lbl"),
        (pl.col("chan_red_lbl_cum") - pl.col("is_red_lbl")).cast(pl.Int16).alias("cnt_prev_red_same_channel"),
        (pl.col("chan_labeled_fb_cum") - pl.col("is_labeled_fb")).cast(pl.Int16).alias("cnt_prev_labeled_same_channel"),
        (pl.col("type_red_lbl_cum") - pl.col("is_red_lbl")).cast(pl.Int16).alias("cnt_prev_red_same_type_lbl"),
        (pl.col("type_labeled_fb_cum") - pl.col("is_labeled_fb")).cast(pl.Int16).alias("cnt_prev_labeled_same_type_lbl"),
        pl.when(pl.col("prev_red_lbl_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_red_lbl_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_red_lbl"),
        pl.when(pl.col("prev_yellow_lbl_ts").is_not_null())
          .then((pl.col("event_ts") - pl.col("prev_yellow_lbl_ts")).dt.total_seconds())
          .otherwise(-1).cast(pl.Int32).alias("sec_since_prev_yellow_lbl"),
    ])

    lf = lf.with_columns([
        pl.when(pl.col("cust_prev_events") > 0)
          .then((pl.col("prev_event_date").is_null() | (pl.col("event_date") != pl.col("prev_event_date"))).cast(pl.Int8))
          .otherwise(1).cast(pl.Int8).alias("_is_new_customer_day"),
        pl.when(pl.col("cust_event_idx") > 2)
          .then(
              ((pl.col("cust_cum_amt_sq") - pl.col("amt") * pl.col("amt")) / (pl.col("cust_event_idx") - 1)
               - pl.col("cust_prev_amt_mean") * pl.col("cust_prev_amt_mean"))
              .clip(lower_bound=0).sqrt())
          .otherwise(0.0).cast(pl.Float32).alias("cust_prev_amt_std"),
        ((pl.col("cust_prev_red_lbl_cnt") + 0.1) / (pl.col("cust_prev_labeled_cnt") + 1.0)).cast(pl.Float32).alias("cust_prev_red_lbl_rate"),
        ((pl.col("cust_prev_yellow_lbl_cnt") + 0.1) / (pl.col("cust_prev_labeled_cnt") + 1.0)).cast(pl.Float32).alias("cust_prev_yellow_lbl_rate"),
        (((pl.col("cust_prev_red_lbl_cnt") + pl.col("cust_prev_yellow_lbl_cnt")) + 0.1) / (pl.col("cust_prev_events") + 1.0)).cast(pl.Float32).alias("cust_prev_susp_lbl_rate"),
        (pl.col("cust_prev_red_lbl_cnt") > 0).cast(pl.Int8).alias("cust_prev_any_red_flag"),
        (pl.col("cust_prev_yellow_lbl_cnt") > 0).cast(pl.Int8).alias("cust_prev_any_yellow_flag"),
        ((pl.col("cnt_prev_red_same_desc_lbl") + 0.1) / (pl.col("cnt_prev_labeled_same_desc") + 1.0)).cast(pl.Float32).alias("red_rate_prev_same_desc_lbl"),
        ((pl.col("cnt_prev_red_same_channel") + 0.1) / (pl.col("cnt_prev_labeled_same_channel") + 1.0)).cast(pl.Float32).alias("red_rate_prev_same_channel"),
        ((pl.col("cnt_prev_red_same_type_lbl") + 0.1) / (pl.col("cnt_prev_labeled_same_type_lbl") + 1.0)).cast(pl.Float32).alias("red_rate_prev_same_type"),
    ])

    lf = lf.with_columns([
        (pl.col("_is_new_customer_day").cum_sum().over("customer_id") - 1).cast(pl.Int16).alias("cust_active_days"),
        pl.when(pl.col("cust_prev_amt_std") > 0)
          .then((pl.col("amt") - pl.col("cust_prev_amt_mean")) / (pl.col("cust_prev_amt_std") + 1.0))
          .otherwise(0.0).cast(pl.Float32).alias("amt_zscore"),
        pl.when(pl.col("cust_prev_max_amt").abs() > 1.0)
          .then(pl.col("amt") / (pl.col("cust_prev_max_amt").abs() + 1.0))
          .otherwise(0.0).cast(pl.Float32).alias("amt_vs_personal_max"),
        pl.when(pl.col("sec_since_prev_event") >= 0)
          .then(pl.col("sec_since_prev_event").cast(pl.Float32) / 86400.0)
          .otherwise(-1.0).cast(pl.Float32).alias("days_since_prev_event"),
        (pl.col("cnt_prev_same_type") == 0).cast(pl.Int8).alias("is_new_event_type"),
        (pl.col("cnt_prev_same_desc") == 0).cast(pl.Int8).alias("is_new_event_desc"),
        (pl.col("cnt_prev_same_subtype") == 0).cast(pl.Int8).alias("is_new_channel_sub"),
        (pl.col("cnt_prev_same_channel_type") == 0).cast(pl.Int8).alias("is_new_channel_type"),
        (pl.col("cnt_prev_same_mcc") == 0).cast(pl.Int8).alias("is_new_mcc"),
        (pl.col("cnt_prev_same_currency") == 0).cast(pl.Int8).alias("is_new_currency"),
        (pl.col("cnt_prev_same_device") == 0).cast(pl.Int8).alias("is_new_device_fp"),
        (pl.col("cnt_prev_same_session") == 0).cast(pl.Int8).alias("is_first_in_session"),
    ])

    lf = lf.with_columns([
        (pl.col("cust_prev_events") / (pl.col("cust_active_days") + 1.0)).cast(pl.Float32).alias("avg_prev_events_per_active_day"),
    ])

    lf = lf.with_columns([
        pl.when(pl.col("is_train_sample")).then((pl.col("train_target_raw") == 1).cast(pl.Int8)).otherwise(pl.lit(None)).alias("target_bin")
    ])

    print(f"  [part {part_id}] computing rolling features...")
    _tmp = lf.select(["event_id", "customer_id", "event_ts", "amt"]).collect().to_pandas()
    _tmp["event_ts"] = pd.to_datetime(_tmp["event_ts"])
    _tmp = _tmp.sort_values(["customer_id", "event_ts", "event_id"]).reset_index(drop=True).set_index("event_ts")

    _results = []
    for _cust, _grp in _tmp.groupby("customer_id", sort=False):
        _s = _grp["amt"]
        _r = pd.DataFrame(index=_grp.index)
        _r["amt_sum_last_1h"]  = _s.rolling("1h",  closed="left").sum()
        _r["cnt_last_1h"]      = _s.rolling("1h",  closed="left").count()
        _r["amt_sum_last_24h"] = _s.rolling("24h", closed="left").sum()
        _r["cnt_last_24h"]     = _s.rolling("24h", closed="left").count()
        _r["max_amt_last_24h"] = _s.rolling("24h", closed="left").max()
        _r["event_id"] = _grp["event_id"].values
        _results.append(_r)

    _roll_df = pd.concat(_results).reset_index(drop=True).fillna(0)
    for c in ["amt_sum_last_1h", "amt_sum_last_24h", "max_amt_last_24h"]:
        _roll_df[c] = _roll_df[c].astype("float32")
    for c in ["cnt_last_1h", "cnt_last_24h"]:
        _roll_df[c] = _roll_df[c].astype("int16")
    _rolling_cols = ["amt_sum_last_1h", "cnt_last_1h", "amt_sum_last_24h", "cnt_last_24h", "max_amt_last_24h"]
    _rolling_lf = pl.from_pandas(_roll_df[["event_id"] + _rolling_cols].drop_duplicates(subset=["event_id"])).lazy()
    lf = lf.join(_rolling_lf, on="event_id", how="left")
    del _tmp, _roll_df, _results, _rolling_lf; gc.collect()

    lf = lf.with_columns([
        pl.when(pl.col("amt_sum_last_1h").abs() > 1.0)
          .then(pl.col("amt") / (pl.col("amt_sum_last_1h").abs() + 1.0))
          .otherwise(0.0).cast(pl.Float32).alias("amt_vs_1h_sum"),
        pl.when(pl.col("amt_sum_last_24h").abs() > 1.0)
          .then(pl.col("amt") / (pl.col("amt_sum_last_24h").abs() + 1.0))
          .otherwise(0.0).cast(pl.Float32).alias("amt_vs_24h_sum"),
    ])

    lf = lf.drop([
        "prev_event_date",
        "prev_same_type_ts", "prev_same_desc_ts",
        "prev_same_channel_type_ts", "prev_same_currency_ts", "prev_same_device_ts",
        "_is_new_customer_day",
    ])

    select_cols = META_COLS + FINAL_FEATURE_COLS
    seen = set()
    select_cols = [c for c in select_cols if not (c in seen or seen.add(c))]
    present = [c for c in select_cols if c in lf.columns]

    out_df = lf.filter(pl.col("is_train_sample") | pl.col("is_test")).select(present).collect()
    out_df.write_parquet(out_path, compression="zstd")
    n_train = int(out_df.filter(pl.col("is_train_sample")).height)
    n_test  = int(out_df.filter(pl.col("is_test")).height)
    print(f"[part {part_id}] done: rows={out_df.height:,}, train_sample={n_train:,}, test={n_test:,}")
    del out_df; gc.collect()
    return out_path


## 3. Сбор базовой train/test-таблицы

In [4]:

feature_paths = []
for part_id in [1, 2, 3]:
    feature_paths.append(build_features_for_part(part_id, force=FORCE_REBUILD_FEATURES))

features = pl.concat([pl.scan_parquet(p) for p in feature_paths], how="vertical_relaxed").collect()
print("Feature table shape:", features.shape)
print("Train sample rows:", features.filter(pl.col("is_train_sample")).height)
print("Test rows:", features.filter(pl.col("is_test")).height)

train_base_df = features.filter(pl.col("is_train_sample")).with_columns(
    pl.col("target_bin").cast(pl.Int8)
).to_pandas()
test_base_df = features.filter(pl.col("is_test")).to_pandas()
del features
gc.collect()

train_base_df["event_ts"] = pd.to_datetime(train_base_df["event_ts"])
test_base_df["event_ts"]  = pd.to_datetime(test_base_df["event_ts"])

train_base_df = train_base_df.sort_values(["event_ts", "event_id"]).reset_index(drop=True)
test_base_df  = test_base_df.sort_values(["event_ts", "event_id"]).reset_index(drop=True)

BASE_FEATURE_COLS = [
    c for c in train_base_df.columns
    if c not in META_COLS and c not in MODEL_DROP_COLS
]
BASE_CAT_COLS = [c for c in CAT_COLS if c in BASE_FEATURE_COLS]
BASE_NUM_COLS = [c for c in BASE_FEATURE_COLS if c not in BASE_CAT_COLS]

print("Base train shape:", train_base_df.shape)
print("Base test shape: ", test_base_df.shape)
print(f"Base features: {len(BASE_FEATURE_COLS)} | cat: {len(BASE_CAT_COLS)} | num: {len(BASE_NUM_COLS)}")


[part 1] building features...
  [part 1] computing rolling features...


/tmp/ipykernel_50211/3632001318.py:7: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = [c for c in BASE_COLS if c in pl.scan_parquet(path).columns]
/tmp/ipykernel_50211/3632001318.py:350: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  present = [c for c in select_cols if c in lf.columns]


[part 1] done: rows=1,744,564, train_sample=1,533,608, test=210,956
[part 2] building features...
  [part 2] computing rolling features...


/tmp/ipykernel_50211/3632001318.py:7: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = [c for c in BASE_COLS if c in pl.scan_parquet(path).columns]
/tmp/ipykernel_50211/3632001318.py:350: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  present = [c for c in select_cols if c in lf.columns]


[part 2] done: rows=1,740,990, train_sample=1,528,941, test=212,049
[part 3] building features...
  [part 3] computing rolling features...


/tmp/ipykernel_50211/3632001318.py:7: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = [c for c in BASE_COLS if c in pl.scan_parquet(path).columns]
/tmp/ipykernel_50211/3632001318.py:350: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  present = [c for c in select_cols if c in lf.columns]


[part 3] done: rows=1,736,693, train_sample=1,526,015, test=210,678
Feature table shape: (5222247, 126)
Train sample rows: 4588564
Test rows: 633683
Base train shape: (4588564, 126)
Base test shape:  (633683, 126)
Base features: 119 | cat: 17 | num: 102


## 4. Causal global priors и device statistics

In [5]:

def dedupe(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def _device_fp_expr_from_raw() -> pl.Expr:
    return (
        (
            pl.col("screen_size").str.extract(r"^(\d+)", 1).cast(pl.Int16, strict=False).fill_null(-1).cast(pl.Int64) * 100_000_000
            + pl.col("screen_size").str.extract(r"x(\d+)$", 1).cast(pl.Int16, strict=False).fill_null(-1).cast(pl.Int64) * 100_000
            + pl.col("operating_system_type").cast(pl.Int16, strict=False).fill_null(-1).cast(pl.Int64) * 1000
            + (pl.col("accept_language").cast(pl.Int32, strict=False).fill_null(-1).cast(pl.Int64) % 1000)
        ).alias("device_fp_i")
    )


PRIOR_COL_DEFS = {
    "event_desc":                 pl.col("event_desc").cast(pl.Int32, strict=False).fill_null(-1).alias("event_desc"),
    "mcc_code_i":                 pl.col("mcc_code").cast(pl.Int32, strict=False).fill_null(-1).alias("mcc_code_i"),
    "timezone":                   pl.col("timezone").cast(pl.Int32, strict=False).fill_null(-1).alias("timezone"),
    "operating_system_type":      pl.col("operating_system_type").cast(pl.Int16, strict=False).fill_null(-1).alias("operating_system_type"),
    "channel_indicator_sub_type": pl.col("channel_indicator_sub_type").cast(pl.Int16, strict=False).fill_null(-1).alias("channel_indicator_sub_type"),
    "event_type_nm":              pl.col("event_type_nm").cast(pl.Int32, strict=False).fill_null(-1).alias("event_type_nm"),
    "pos_cd":                     pl.col("pos_cd").cast(pl.Int16, strict=False).fill_null(-1).alias("pos_cd"),
    "channel_indicator_type":     pl.col("channel_indicator_type").cast(pl.Int16, strict=False).fill_null(-1).alias("channel_indicator_type"),
    "accept_language_i":          pl.col("accept_language").cast(pl.Int32, strict=False).fill_null(-1).alias("accept_language_i"),
    "browser_language_i":         pl.col("browser_language").cast(pl.Int32, strict=False).fill_null(-1).alias("browser_language_i"),
    "device_fp_i":                _device_fp_expr_from_raw(),
}

DEVICE_STAT_COLS = [
    "device_n_distinct_customers",
    "device_n_distinct_sessions",
    "device_n_total_ops",
]


def _scan_train_prior_source(expr: pl.Expr, key_name: str, cutoff: pd.Timestamp | None = None) -> pl.LazyFrame:
    frames = []
    for i in [1, 2, 3]:
        lf = pl.scan_parquet(DATA_DIR / f"train_part_{i}.parquet").select([
            pl.col("event_id"),
            pl.col("event_dttm").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False).alias("event_ts"),
            expr,
        ])
        if cutoff is not None:
            lf = lf.filter(pl.col("event_ts") < pl.lit(cutoff.to_pydatetime()))
        frames.append(lf.select(["event_id", key_name]))
    return pl.concat(frames, how="vertical_relaxed")


def _make_prior_from_source(lf: pl.LazyFrame, key_name: str) -> pl.DataFrame:
    cnt_col = f"prior_{key_name}_cnt"
    lbl_cnt_col = f"prior_{key_name}_lbl_cnt"
    red_cnt_col = f"prior_{key_name}_red_cnt"

    total = lf.group_by(key_name).len().rename({"len": cnt_col})
    labeled = (
        lf.join(labels_lf, on="event_id", how="inner")
          .group_by(key_name)
          .agg([
              pl.len().alias(lbl_cnt_col),
              pl.sum("target").cast(pl.Float64).alias(red_cnt_col),
          ])
    )

    return (
        total.join(labeled, on=key_name, how="left")
             .with_columns([
                 pl.col(lbl_cnt_col).fill_null(0.0),
                 pl.col(red_cnt_col).fill_null(0.0),
             ])
             .with_columns([
                 ((pl.col(red_cnt_col) + 1.0) / (pl.col(cnt_col) + 200.0)).cast(pl.Float32).alias(f"prior_{key_name}_red_rate_all"),
                 ((pl.col(lbl_cnt_col) + 1.0) / (pl.col(cnt_col) + 200.0)).cast(pl.Float32).alias(f"prior_{key_name}_labeled_rate_all"),
                 ((pl.col(red_cnt_col) + 1.0) / (pl.col(lbl_cnt_col) + 2.0)).cast(pl.Float32).alias(f"prior_{key_name}_red_share_labeled"),
             ])
             .select([
                 key_name,
                 cnt_col,
                 f"prior_{key_name}_red_rate_all",
                 f"prior_{key_name}_labeled_rate_all",
                 f"prior_{key_name}_red_share_labeled",
             ])
             .collect()
    )


def build_prior_tables(tag: str, cutoff: pd.Timestamp | None = None, force: bool = False) -> Tuple[Dict[str, pd.DataFrame], List[str]]:
    tables: Dict[str, pd.DataFrame] = {}
    prior_feature_cols: List[str] = []

    for key_name, expr in PRIOR_COL_DEFS.items():
        out_path = CACHE_DIR / f"prior_{GLOBAL_CACHE_TAG}_{tag}_{key_name}.parquet"
        if out_path.exists() and not force:
            prior_df = pl.read_parquet(out_path)
        else:
            print(f"Building prior table: {tag} | {key_name}")
            source_lf = _scan_train_prior_source(expr, key_name, cutoff=cutoff)
            prior_df = _make_prior_from_source(source_lf, key_name)
            prior_df.write_parquet(out_path)

        tables[key_name] = prior_df.to_pandas()
        prior_feature_cols.extend([c for c in prior_df.columns if c != key_name])

    return tables, dedupe(prior_feature_cols)


def _scan_device_history(include_pretest: bool, cutoff: pd.Timestamp | None = None) -> pl.LazyFrame:
    base_cols = [
        "customer_id", "event_dttm", "session_id",
        "screen_size", "operating_system_type", "accept_language",
    ]
    frames = []
    for i in [1, 2, 3]:
        frames.append(pl.scan_parquet(DATA_DIR / f"pretrain_part_{i}.parquet").select(base_cols))
        frames.append(pl.scan_parquet(DATA_DIR / f"train_part_{i}.parquet").select(base_cols))

    if include_pretest:
        frames.append(pl.scan_parquet(DATA_DIR / "pretest.parquet").select(base_cols))

    lf = pl.concat(frames, how="diagonal_relaxed").with_columns([
        pl.col("event_dttm").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False).alias("event_ts"),
        pl.col("session_id").cast(pl.Int64, strict=False).fill_null(-1).alias("session_id"),
        _device_fp_expr_from_raw(),
    ])

    if cutoff is not None:
        lf = lf.filter(pl.col("event_ts") < pl.lit(cutoff.to_pydatetime()))

    return lf.select(["device_fp_i", "customer_id", "session_id"])


def build_device_stats_table(tag: str, cutoff: pd.Timestamp | None = None, include_pretest: bool = False, force: bool = False) -> pd.DataFrame:
    out_path = CACHE_DIR / f"device_stats_{GLOBAL_CACHE_TAG}_{tag}.parquet"
    if out_path.exists() and not force:
        return pl.read_parquet(out_path).to_pandas()

    print(f"Building device stats: {tag}")
    lf = _scan_device_history(include_pretest=include_pretest, cutoff=cutoff)
    stats_df = (
        lf.group_by("device_fp_i")
          .agg([
              pl.n_unique("customer_id").cast(pl.Int32).alias("device_n_distinct_customers"),
              pl.n_unique("session_id").cast(pl.Int32).alias("device_n_distinct_sessions"),
              pl.len().cast(pl.Int32).alias("device_n_total_ops"),
          ])
          .with_columns([
              pl.col("device_n_distinct_customers").log1p().cast(pl.Float32).alias("device_n_distinct_customers"),
              pl.col("device_n_distinct_sessions").log1p().cast(pl.Float32).alias("device_n_distinct_sessions"),
              pl.col("device_n_total_ops").log1p().cast(pl.Float32).alias("device_n_total_ops"),
          ])
          .collect()
    )
    stats_df.write_parquet(out_path)
    return stats_df.to_pandas()


def attach_global_features(base_df: pd.DataFrame, device_stats_df: pd.DataFrame, prior_tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    out = base_df.merge(device_stats_df, on="device_fp_i", how="left")
    for key_name, prior_df in prior_tables.items():
        out = out.merge(prior_df, on=key_name, how="left")
    return out


def audit_and_select_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: List[str],
    cat_cols: List[str],
) -> Tuple[List[str], pd.DataFrame]:
    rows = []
    keep_cols = []

    for col in feature_cols:
        if col not in train_df.columns or col not in test_df.columns:
            rows.append({
                "feature": col,
                "drop_reason": "missing_in_split",
                "train_nunique": np.nan,
                "test_nunique": np.nan,
                "train_missing_rate": np.nan,
                "test_missing_rate": np.nan,
                "dominant_share": np.nan,
            })
            continue

        s_tr = train_df[col]
        s_te = test_df[col]

        train_nunique = int(s_tr.nunique(dropna=False))
        test_nunique = int(s_te.nunique(dropna=False))
        train_missing_rate = float(s_tr.isna().mean())
        test_missing_rate = float(s_te.isna().mean())

        dominant_share = 0.0
        if (col in cat_cols) or (train_nunique <= 128) or (train_missing_rate >= 0.95):
            vc = s_tr.astype("object").value_counts(dropna=False, normalize=True)
            dominant_share = float(vc.iloc[0]) if len(vc) else 1.0

        drop_reason = "keep"
        if col in RISKY_LABEL_FEATURES:
            drop_reason = "risky_label_feature"
        elif any(col.endswith(sfx) for sfx in RISKY_PRIOR_SUFFIXES):
            drop_reason = "risky_label_prior"
        elif train_nunique <= 1:
            drop_reason = "constant"
        elif dominant_share >= 0.9995:
            drop_reason = "almost_constant"
        elif train_missing_rate >= 0.995 and test_missing_rate >= 0.995:
            drop_reason = "too_sparse"
        elif (col in cat_cols) and (train_missing_rate >= 0.85) and (train_nunique >= 20) and (test_nunique <= 2):
            drop_reason = "collapsed_in_test"

        rows.append({
            "feature": col,
            "drop_reason": drop_reason,
            "train_nunique": train_nunique,
            "test_nunique": test_nunique,
            "train_missing_rate": train_missing_rate,
            "test_missing_rate": test_missing_rate,
            "dominant_share": dominant_share,
        })

        if drop_reason == "keep":
            keep_cols.append(col)

    audit_df = pd.DataFrame(rows).sort_values(
        ["drop_reason", "train_missing_rate", "dominant_share", "feature"],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)
    return keep_cols, audit_df


def prepare_feature_matrices(
    fit_df: pd.DataFrame,
    apply_dfs: List[pd.DataFrame],
    feature_cols: List[str],
    cat_cols: List[str],
) -> Tuple[pd.DataFrame, ...]:
    fit_X = fit_df[feature_cols].copy()
    apply_Xs = [df[feature_cols].copy() for df in apply_dfs]

    for c in cat_cols:
        fit_X[c] = fit_X[c].fillna(-1).astype(np.int64)
        for X in apply_Xs:
            X[c] = X[c].fillna(-1).astype(np.int64)

    num_cols = [c for c in feature_cols if c not in cat_cols]
    if num_cols:
        medians = fit_X[num_cols].median(numeric_only=True)
        fit_X[num_cols] = fit_X[num_cols].fillna(medians)
        for X in apply_Xs:
            X[num_cols] = X[num_cols].fillna(medians)

    return (fit_X, *apply_Xs)


## 5. Локальная holdout-валидация без leakage

In [6]:

print("Building causal globals for local validation...")
device_stats_local = build_device_stats_table(
    tag="pre_val",
    cutoff=VAL_START,
    include_pretest=False,
    force=FORCE_REBUILD_GLOBALS,
)
prior_tables_local, prior_feature_cols = build_prior_tables(
    tag="pre_val",
    cutoff=VAL_START,
    force=FORCE_REBUILD_GLOBALS,
)

train_local_df = attach_global_features(train_base_df, device_stats_local, prior_tables_local)
test_local_df = attach_global_features(test_base_df, device_stats_local, prior_tables_local)
del device_stats_local, prior_tables_local
gc.collect()

raw_feature_cols = dedupe(BASE_FEATURE_COLS + DEVICE_STAT_COLS + prior_feature_cols)
raw_cat_feature_cols = [c for c in CAT_COLS if c in raw_feature_cols]

FEATURE_COLS, feature_audit_df = audit_and_select_features(
    train_local_df,
    test_local_df,
    raw_feature_cols,
    raw_cat_feature_cols,
)
CAT_FEATURE_COLS = [c for c in CAT_COLS if c in FEATURE_COLS]
NUM_FEATURE_COLS = [c for c in FEATURE_COLS if c not in CAT_FEATURE_COLS]

val_mask = train_local_df["event_ts"] >= VAL_START

print("Local train rows:", int((~val_mask).sum()))
print("Local val rows:  ", int(val_mask.sum()))
print(f"Features after cleaning: {len(FEATURE_COLS)} | cat: {len(CAT_FEATURE_COLS)} | num: {len(NUM_FEATURE_COLS)}")
print("Dropped features by reason:")
print(feature_audit_df.loc[feature_audit_df["drop_reason"] != "keep", "drop_reason"].value_counts())

feature_audit_df[feature_audit_df["drop_reason"] != "keep"].head(20)


Building causal globals for local validation...
Local train rows: 3319004
Local val rows:   1269560
Features after cleaning: 118 | cat: 15 | num: 103
Dropped features by reason:
drop_reason
risky_label_prior      33
risky_label_feature    10
constant                4
almost_constant         1
Name: count, dtype: int64


,feature,drop_reason,train_nunique,test_nunique,train_missing_rate,test_missing_rate,dominant_share
0,is_new_currency,almost_constant,2,2,0.000000e+00,0.000000,0.999901
1,accept_language_i,constant,1,1,0.000000e+00,0.000000,1.000000
2,browser_language_i,constant,1,1,0.000000e+00,0.000000,1.000000
3,prior_accept_language_i_cnt,constant,1,1,0.000000e+00,0.000000,1.000000
4,prior_browser_language_i_cnt,constant,1,1,0.000000e+00,0.000000,1.000000
123,cnt_prev_red_same_desc_lbl,risky_label_feature,57,30,0.000000e+00,0.000000,0.958752
124,cnt_prev_yellow_same_desc_lbl,risky_label_feature,7,7,0.000000e+00,0.000000,0.954000
125,cnt_prev_red_same_type_lbl,risky_label_feature,85,34,0.000000e+00,0.000000,0.938541
126,cnt_prev_red_same_channel,risky_label_feature,114,32,0.000000e+00,0.000000,0.930188
127,cnt_prev_labeled_same_desc,risky_label_feature,57,31,0.000000e+00,0.000000,0.913195


In [7]:

def make_weights(raw_target: np.ndarray, event_ts=None) -> np.ndarray:
    w = np.where(raw_target == 1, 10.0, np.where(raw_target == 0, 2.5, 1.0)).astype(np.float32)
    if event_ts is not None:
        ts = pd.to_datetime(event_ts)
        border = pd.Timestamp(NEG_SAMPLE_BORDER_STR)
        g = raw_target == -1
        w[g & (ts >= border)] = 1.8
        w[g & (ts < border)]  = 3.0
        w[ts >= RECENCY_WEIGHT_START] *= 1.15
    return w


def fit_catboost_with_holdout(X_tr, y_tr, w_tr, X_val, y_val, w_val, cat_cols, params, use_gpu=True):
    params = params.copy()
    params.update({
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": RANDOM_SEED,
        "allow_writing_files": False,
        "verbose": 200,
        "metric_period": 100,
    })

    if use_gpu:
        params.update({"task_type": "GPU", "devices": "0"})
    else:
        params.update({"task_type": "CPU", "thread_count": max(1, (os.cpu_count() or 4) - 1)})

    tr_pool = Pool(X_tr, y_tr, weight=w_tr, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, weight=w_val, cat_features=cat_cols)

    def _fit_with_params(_params):
        model = CatBoostClassifier(**_params)
        model.fit(tr_pool, eval_set=val_pool, use_best_model=True)
        return model

    try:
        model = _fit_with_params(params)
    except Exception as e:
        print("CatBoost fallback:", e)
        params.pop("devices", None)
        params["task_type"] = "CPU"
        params["thread_count"] = max(1, (os.cpu_count() or 4) - 1)
        try:
            model = _fit_with_params(params)
        except Exception as e2:
            print("CatBoost PRAUC fallback -> AUC:", e2)
            params["eval_metric"] = "AUC"
            model = _fit_with_params(params)

    val_raw = model.predict(val_pool, prediction_type="RawFormulaVal")
    val_ap = average_precision_score(y_val, val_raw)

    best_iter = model.get_best_iteration()
    if best_iter is None:
        best_iter = int(params.get("iterations", 1000))
    else:
        best_iter = int(best_iter) + 1

    print(f"CB best_iter={best_iter}, val_pr_auc={val_ap:.6f}")
    return model, int(best_iter), float(val_ap), params


def refit_full_catboost(X, y, w, cat_cols, base_params, best_iter):
    params = {k: v for k, v in base_params.items() if k not in ("od_type", "od_wait")}
    params["iterations"] = int(max(100, round(best_iter * REFIT_ITER_MULT)))

    y_arr = np.asarray(y)
    w_arr = np.asarray(w, dtype=np.float32)
    if w_arr.ndim == 0 or w_arr.shape[0] != len(y_arr):
        w_arr = np.full(len(y_arr), float(np.nanmean(w_arr)) if w_arr.size > 0 else 1.0, dtype=np.float32)

    model = CatBoostClassifier(**params)
    model.fit(Pool(X, y_arr, weight=w_arr, cat_features=cat_cols), verbose=200)
    return model


def _lgb_prepare(X: pd.DataFrame, cat_cols: List[str]) -> pd.DataFrame:
    Xc = X.copy()
    for col in cat_cols:
        Xc[col] = Xc[col].fillna(-1).astype(np.int64) + 1
    return Xc


def _lgb_ap_metric(preds, dataset):
    y_true = dataset.get_label()
    return "ap", average_precision_score(y_true, preds), True


def fit_lgb_with_holdout(X_tr, y_tr, w_tr, X_val, y_val, w_val, cat_cols, params):
    p = params.copy()
    n_estimators = int(p.pop("n_estimators", 5000))
    early_stopping_rounds = int(p.pop("early_stopping_rounds", 200))
    p.update({
        "objective": "binary",
        "metric": "None",
        "seed": RANDOM_SEED,
        "verbosity": -1,
        "n_jobs": -1,
        "feature_pre_filter": False,
    })

    Xtr_ = _lgb_prepare(X_tr, cat_cols)
    Xval_ = _lgb_prepare(X_val, cat_cols)

    tr_ds = lgb.Dataset(
        Xtr_,
        label=y_tr,
        weight=w_tr,
        categorical_feature=cat_cols,
        free_raw_data=False,
    )
    val_ds = lgb.Dataset(
        Xval_,
        label=y_val,
        weight=w_val,
        reference=tr_ds,
        categorical_feature=cat_cols,
        free_raw_data=False,
    )

    callbacks = [
        lgb.early_stopping(early_stopping_rounds, first_metric_only=True, verbose=False),
        lgb.log_evaluation(200),
    ]

    model = lgb.train(
        p,
        tr_ds,
        num_boost_round=n_estimators,
        valid_sets=[val_ds],
        valid_names=["valid"],
        feval=_lgb_ap_metric,
        callbacks=callbacks,
    )

    best_iter = model.best_iteration or n_estimators
    val_raw = model.predict(Xval_, num_iteration=best_iter)
    val_ap = average_precision_score(y_val, val_raw)
    print(f"LGB best_iter={best_iter}, val_pr_auc={val_ap:.6f}")
    return model, int(best_iter), float(val_ap), p


def refit_full_lgb(X, y, w, cat_cols, base_params, best_iter):
    p = {k: v for k, v in base_params.items() if k not in ("n_estimators", "early_stopping_rounds")}
    p.update({
        "objective": "binary",
        "metric": "None",
        "seed": RANDOM_SEED,
        "verbosity": -1,
        "n_jobs": -1,
        "feature_pre_filter": False,
    })

    X_ = _lgb_prepare(X, cat_cols)
    ds = lgb.Dataset(
        X_,
        label=y,
        weight=w,
        categorical_feature=cat_cols,
        free_raw_data=False,
    )
    model = lgb.train(p, ds, num_boost_round=int(max(100, round(best_iter * REFIT_ITER_MULT))))
    return model


def rank_norm(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    return rankdata(x, method="average") / (len(x) + 1.0)


def optimize_blend_weights(heads: Dict[str, np.ndarray], y_true: np.ndarray):
    keys = list(heads.keys())
    preds = [heads[k] for k in keys]

    def neg_ap(logits):
        w = softmax(logits)
        blend = sum(w[i] * preds[i] for i in range(len(keys)))
        return -average_precision_score(y_true, blend)

    x0 = np.zeros(len(keys), dtype=np.float64)
    res = minimize(
        neg_ap,
        x0,
        method="Nelder-Mead",
        options={"maxiter": 1500, "xatol": 1e-4, "fatol": 1e-6},
    )

    best_w = softmax(res.x).astype(np.float32)
    best_ap = float(-res.fun)
    return keys, best_w, best_ap


In [8]:

main_train_df = train_local_df.loc[~val_mask].reset_index(drop=True)
main_val_df   = train_local_df.loc[val_mask].reset_index(drop=True)

print("Main train rows:", len(main_train_df), "| val:", len(main_val_df))

X_main_tr, X_main_val = prepare_feature_matrices(
    main_train_df,
    [main_val_df],
    FEATURE_COLS,
    CAT_FEATURE_COLS,
)

y_main_tr = main_train_df["target_bin"].astype(np.int8).values
y_main_val = main_val_df["target_bin"].astype(np.int8).values
w_main_tr = make_weights(main_train_df["train_target_raw"].values, main_train_df["event_ts"].values)
w_main_val = make_weights(main_val_df["train_target_raw"].values, main_val_df["event_ts"].values)

params_cb = {
    "iterations": 6000,
    "learning_rate": 0.04,
    "depth": 8,
    "l2_leaf_reg": 10.0,
    "random_strength": 0.5,
    "od_type": "Iter",
    "od_wait": 300,
}

params_lgb = {
    "n_estimators": 6000,
    "early_stopping_rounds": 300,
    "learning_rate": 0.03,
    "num_leaves": 255,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.75,
    "reg_alpha": 0.2,
    "reg_lambda": 8.0,
    "min_split_gain": 0.0,
    "max_bin": 255,
    "device": "cpu",
}

print("Fitting CatBoost...")
model_cb, best_iter_cb, ap_cb, used_cb = fit_catboost_with_holdout(
    X_main_tr, y_main_tr, w_main_tr,
    X_main_val, y_main_val, w_main_val,
    cat_cols=CAT_FEATURE_COLS,
    params=params_cb,
    use_gpu=USE_GPU,
)

print("Fitting LightGBM...")
model_lgb, best_iter_lgb, ap_lgb, used_lgb = fit_lgb_with_holdout(
    X_main_tr, y_main_tr, w_main_tr,
    X_main_val, y_main_val, w_main_val,
    cat_cols=CAT_FEATURE_COLS,
    params=params_lgb,
)

val_pool_main = Pool(X_main_val, y_main_val, cat_features=CAT_FEATURE_COLS)

pred_cb = model_cb.predict(val_pool_main, prediction_type="RawFormulaVal")
pred_lgb = model_lgb.predict(_lgb_prepare(X_main_val, CAT_FEATURE_COLS), num_iteration=best_iter_lgb)

heads = {
    "catboost": rank_norm(pred_cb),
    "lightgbm": rank_norm(pred_lgb),
}

for name, pred in heads.items():
    print(f"{name:<10} val PR-AUC: {average_precision_score(y_main_val, pred):.6f}")

blend_keys, best_w, best_ap = optimize_blend_weights(heads, y_main_val)
print("\nBest weights:", {blend_keys[i]: round(float(best_w[i]), 4) for i in range(len(blend_keys))})
print(f"Best blended val PR-AUC: {best_ap:.6f}")


Main train rows: 3319004 | val: 1269560
Fitting CatBoost...


Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.6936719	test: 0.6719140	best: 0.6719140 (0)	total: 398ms	remaining: 39m 45s
200:	learn: 0.8954605	test: 0.7059112	best: 0.7059112 (200)	total: 1m 17s	remaining: 37m 25s
400:	learn: 0.9242744	test: 0.7053164	best: 0.7075486 (262)	total: 2m 40s	remaining: 37m 21s
bestTest = 0.7075485602
bestIteration = 262
Shrink model to first 263 iterations.
CB best_iter=263, val_pr_auc=0.447587
Fitting LightGBM...


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



[200]	valid's ap: 0.482131
[400]	valid's ap: 0.482085
[600]	valid's ap: 0.479113
LGB best_iter=307, val_pr_auc=0.483563
catboost   val PR-AUC: 0.447587
lightgbm   val PR-AUC: 0.483563

Best weights: {'catboost': 0.0674, 'lightgbm': 0.9326}
Best blended val PR-AUC: 0.483993


## 6. Refit на full train и submission

In [9]:

print("Building causal globals for full fit / test...")
device_stats_full = build_device_stats_table(
    tag="full_history",
    cutoff=None,
    include_pretest=True,
    force=FORCE_REBUILD_GLOBALS,
)
prior_tables_full, prior_feature_cols_full = build_prior_tables(
    tag="full_train",
    cutoff=None,
    force=FORCE_REBUILD_GLOBALS,
)

train_full_df = attach_global_features(train_base_df, device_stats_full, prior_tables_full)
test_full_df = attach_global_features(test_base_df, device_stats_full, prior_tables_full)
del device_stats_full, prior_tables_full
gc.collect()

FULL_FEATURE_COLS = [c for c in FEATURE_COLS if c in train_full_df.columns and c in test_full_df.columns]
FULL_CAT_FEATURE_COLS = [c for c in CAT_COLS if c in FULL_FEATURE_COLS]

X_main_full, X_test_main = prepare_feature_matrices(
    train_full_df,
    [test_full_df],
    FULL_FEATURE_COLS,
    FULL_CAT_FEATURE_COLS,
)

y_main_full = train_full_df["target_bin"].astype(np.int8).values
w_main_full = make_weights(train_full_df["train_target_raw"].values, train_full_df["event_ts"].values)

if RETRAIN_ON_FULL:
    print("Refit CatBoost...")
    model_cb_f = refit_full_catboost(
        X_main_full, y_main_full, w_main_full,
        FULL_CAT_FEATURE_COLS, used_cb, best_iter_cb,
    )

    print("Refit LightGBM...")
    model_lgb_f = refit_full_lgb(
        X_main_full, y_main_full, w_main_full,
        FULL_CAT_FEATURE_COLS, used_lgb, best_iter_lgb,
    )
else:
    model_cb_f = model_cb
    model_lgb_f = model_lgb

test_pool_main = Pool(X_test_main, cat_features=FULL_CAT_FEATURE_COLS)

tp_cb = model_cb_f.predict(test_pool_main, prediction_type="RawFormulaVal")
tp_lgb = model_lgb_f.predict(_lgb_prepare(X_test_main, FULL_CAT_FEATURE_COLS))

test_heads = {
    "catboost": rank_norm(tp_cb),
    "lightgbm": rank_norm(tp_lgb),
}
test_pred_blend = sum(best_w[i] * test_heads[blend_keys[i]] for i in range(len(blend_keys)))

pred_df = pd.DataFrame({
    "event_id": test_full_df["event_id"].values,
    "predict": test_pred_blend,
})

sample_submit = pd.read_csv(DATA_DIR / "sample_submit.csv")
submission = sample_submit[["event_id"]].merge(pred_df, on="event_id", how="left")

missing = int(submission["predict"].isna().sum())
print("Submission rows:", len(submission), "Missing:", missing)
assert missing == 0, "Some test event_id are missing in predictions"

sub_path = CACHE_DIR / "submission_dual_boost.csv"
submission.to_csv(sub_path, index=False)

meta = {
    "feature_count": len(FULL_FEATURE_COLS),
    "cat_feature_count": len(FULL_CAT_FEATURE_COLS),
    "blend_keys": blend_keys,
    "blend_weights": {blend_keys[i]: float(best_w[i]) for i in range(len(blend_keys))},
    "local_val_ap": float(best_ap),
    "head_scores": {
        "catboost": float(ap_cb),
        "lightgbm": float(ap_lgb),
    },
    "best_iters": {
        "catboost": int(best_iter_cb),
        "lightgbm": int(best_iter_lgb),
    },
}
meta_path = CACHE_DIR / "dual_boost_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved ->", sub_path)
print("Saved ->", meta_path)
submission.head()


Building causal globals for full fit / test...
Refit CatBoost...


Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.6923139	total: 529ms	remaining: 2m 32s
200:	learn: 0.8886337	total: 1m 2s	remaining: 27.6s
288:	learn: 0.9053549	total: 1m 31s	remaining: 0us
Refit LightGBM...


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negativ

,event_id,predict
0,125854726334416,0.580603
1,125949211749418,0.645864
2,124437385134670,0.613026
3,124394437682654,0.609971
4,123973531121838,0.536586
